# Анализ датасета дата-центров по странам мира


#### __1. Формулировка бизнес-задачи__

__Бизнес-задача:__ Международная компания планирует расширение своей инфраструктуры дата-центров и нуждается в анализе текущего рынка дата-центров по странам для определения наиболее перспективных локаций для инвестиций.

__Цель анализа:__ Выявить ключевые факторы, влияющие на развитие дата-центров в разных странах, и определить наиболее привлекательные рынки для инвестирования.

#### __2. Гипотезы о факторах, влияющих на решение задачи__

- **Гипотеза 1:** Страны с высоким уровнем интернет-проникновения имеют больше дата-центров
- **Гипотеза 2:** Использование возобновляемой энергии положительно коррелирует с количеством дата-центров
- **Гипотеза 3:** Низкая задержка (latency) до глобальных хабов способствует развитию дата-центров
- **Гипотеза 4:** Наличие крупных облачных провайдеров стимулирует строительство дата-центров
- **Гипотеза 5:** Страны с большей площадью дата-центров имеют большую мощность

#### __3. Определение необходимых и доступных данных__
 
 __Необходимые данные:__
 - Количество дата-центров по странам
 - Мощность дата-центров
 - Использование возобновляемой энергии
 - Интернет-проникновение
 - Задержка до глобальных хабов
 - Наличие облачных провайдеров
 
 __Доступные данные:__ Используется датасет Book1.xlsx, содержащий данные о дата-центрах по странам мира, включая показатели мощности, энергопотребления, интернет-инфраструктуры и наличия облачных провайдеров.
 
##### __Номинальная шкала:__
- country (Название страны)
- key_operators (Ключевые операторы дата-центров)
- cloud_provider (Наличие облачных провайдеров)
- cooling_technologies_common (Технологии охлаждения)
- regulatory_challenges_or_limits (Регуляторные ограничения)
- green_dc_initiatives_description (Описание "зеленых" инициатив)

##### __Шкала отношений:__

###### __Исходные:__
- total_data_centers (Общее количество дата-центров)
- hyperscale_data_centers (Количество гиперскейл-дата-центров)
- colocation_data_centers (Количество колокейшн-дата-центров)
- floor_space_sqft_total (Общая площадь в кв. футах)
- power_capacity_MW_total (Общая мощность в МВт)
- average_renewable_energy_usage_percent (Средний процент использования возобновляемой энергии)
- internet_penetration_percent (Процент интернет-проникновения)
- avg_latency_to_global_hubs_ms (Средняя задержка до глобальных хабов в мс)
- number_of_fiber_connections (Количество оптоволоконных соединений)
- growth_rate_of_data_centers_percent_per_year (Темп роста дата-центров в процентах в год)

###### __Производные (после вычислений):__
- tier_3_percentage (Процент дата-центров Tier III уровня)
- total_cloud_providers (Общее количество облачных провайдеров)
- has_aws / has_azure / has_gcp (Наличие конкретных облачных провайдеров)
- cooling_air / cooling_liquid / cooling_immersion / cooling_free (Наличие конкретных технологий охлаждения)

###### __Трансформированные:__
- total_data_centers_std (стандартизированное количество дата-центров)
- power_capacity_MW_norm (нормализованная мощность)
- renewable_energy_percent_std (стандартизированный процент возобновляемой энергии)

#### __4. Загрузка и подготовка набора данных__

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Загрузка данных
df = pd.read_excel('Data\Dataset.xlsx', sheet_name='Sheet1')

In [4]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df.shape)

print("\nПропуски в данных:")
print(df.isnull().sum())


Размер датасета: (191, 21)

Пропуски в данных:
country                                           0
total_data_centers                                0
hyperscale_data_centers                           0
colocation_data_centers                           0
floor_space_sqft_total                            0
power_capacity_MW_total                           0
average_renewable_energy_usage_percent            0
tier_distribution                                12
key_operators                                    21
cloud_provider                                   42
internet_penetration_percent                      0
avg_latency_to_global_hubs_ms                     0
number_of_fiber_connections                       0
growth_rate_of_data_centers_percent_per_year      2
cooling_technologies_common                       1
regulatory_challenges_or_limits                   0
disaster_recovery_sites_common                   13
green_dc_initiatives_description                 26
source_of_data  

In [6]:
# Проверка дубликатов
print(f"Количество дубликатов: {df.duplicated().sum()}")

Количество дубликатов: 0


In [8]:
# осмотр типов данных
print(df.dtypes)

country                                         object
total_data_centers                               int64
hyperscale_data_centers                         object
colocation_data_centers                         object
floor_space_sqft_total                          object
power_capacity_MW_total                         object
average_renewable_energy_usage_percent          object
tier_distribution                               object
key_operators                                   object
cloud_provider                                  object
internet_penetration_percent                    object
avg_latency_to_global_hubs_ms                   object
number_of_fiber_connections                     object
growth_rate_of_data_centers_percent_per_year    object
cooling_technologies_common                     object
regulatory_challenges_or_limits                 object
disaster_recovery_sites_common                  object
green_dc_initiatives_description                object
source_of_

#### __5. Очистка данных__

In [9]:
# копия для очистки
df_clean = df.copy()

In [11]:
# Функция для очистки строк с числами
def clean_numeric_string(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return value
    # Удаляем символы ~, +, %, пробелы и другие
    cleaned = str(value).replace('~', '').replace('+', '').replace('%', '').replace(',', '').strip()
    # Если после очистки пусто, возвращаем NaN
    if cleaned == '' or cleaned == 'Unknown' or 'Unknown' in str(value):
        return np.nan
    try:
        return float(cleaned)
    except:
        return np.nan
    
# Список колонок для очистки
columns_to_clean = [
    'total_data_centers', 'hyperscale_data_centers', 'colocation_data_centers',
    'floor_space_sqft_total', 'power_capacity_MW_total', 'average_renewable_energy_usage_percent',
    'internet_penetration_percent', 'avg_latency_to_global_hubs_ms', 
    'number_of_fiber_connections', 'growth_rate_of_data_centers_percent_per_year'
]

# Очистка числовых колонок
for col in columns_to_clean:
    if col in df_clean.columns:
        df_clean[col + '_clean'] = df_clean[col].apply(clean_numeric_string)

In [13]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df_clean.shape)

print("\nПропуски в данных:")
print(df_clean.isnull().sum())


Размер датасета: (191, 31)

Пропуски в данных:
country                                                 0
total_data_centers                                      0
hyperscale_data_centers                                 0
colocation_data_centers                                 0
floor_space_sqft_total                                  0
power_capacity_MW_total                                 0
average_renewable_energy_usage_percent                  0
tier_distribution                                      12
key_operators                                          21
cloud_provider                                         42
internet_penetration_percent                            0
avg_latency_to_global_hubs_ms                           0
number_of_fiber_connections                             0
growth_rate_of_data_centers_percent_per_year            2
cooling_technologies_common                             1
regulatory_challenges_or_limits                         0
disaster_recovery_sites_

#### __5. Определение шкалы измерения признаков__